## Useful Links

- Concurrency Models Theory --> https://medium.com/data-science/deep-dive-into-multithreading-multiprocessing-and-asyncio-94fdbe0c91f0
- Asyncio --> https://realpython.com/async-io-python/

**How the operating System (OS) manages threads and processes** 

---

### 🔹 **What is a Process?**
A **process** is an independent program in execution. Each process:
- Has its own memory space (code, data, stack, heap).
- Has at least one thread (called the main thread).
- Is isolated from other processes.

---

### 🔹 **What is a Thread?**
A **thread** is the smallest unit of execution within a process.
- All threads within a process share the **same memory space**.
- Threads are lighter than processes in terms of creation and context switching overhead.

---

### 🧠 **How the OS Manages Processes**
1. **Creation & Termination**:
   - Processes are created using system calls like `fork()` (Unix) or `CreateProcess()` (Windows).
   - Terminated using `exit()` or by signals (e.g., `SIGKILL`).

2. **Scheduling**:
   - OS uses **schedulers** to decide which process gets CPU time.
   - Popular algorithms: Round Robin, Multilevel Queue, Shortest Job First, etc.

3. **Context Switching**:
   - When switching from one process to another, the OS saves the state (registers, program counter) of the current process and loads the state of the next one.
   - This is expensive due to memory isolation between processes.

4. **Inter-Process Communication (IPC)**:
   - Since processes don’t share memory, IPC mechanisms like pipes, message queues, shared memory, or sockets are used.

5. **Process States**:
   - **New** → **Ready** → **Running** → **Waiting** → **Terminated**
   - OS maintains a **Process Control Block (PCB)** for each process to track its state, registers, memory, priority, etc.

---

### 🧵 **How the OS Manages Threads**
1. **Types of Threads**:
   - **User-level Threads**: Managed by user libraries; OS isn’t aware.
   - **Kernel-level Threads**: Managed directly by the OS.
   - **Hybrid Model**: Many user threads mapped to many kernel threads (e.g., Linux NPTL).

2. **Thread Scheduling**:
   - Kernel-level threads are scheduled just like processes.
   - Thread context switch is cheaper than process switch since memory is shared.

3. **Thread Synchronization**:
   - Threads in the same process share data — leads to **race conditions**.
   - OS provides mutexes, semaphores, condition variables, etc., to prevent data corruption.

4. **Thread Control Block (TCB)**:
   - Just like processes have PCBs, threads have TCBs which store register states, thread ID, priority, and stack pointer.

---

### 🔁 **Process vs Thread: OS View**
| Feature          | Process                  | Thread                     |
|------------------|---------------------------|-----------------------------|
| Memory           | Own memory                | Shares memory with others  |
| Overhead         | High (heavyweight)        | Low (lightweight)          |
| Communication    | Through IPC               | Direct memory access       |
| Creation time    | More time                 | Less time                  |
| Crash impact     | Isolated                  | Can crash entire process   |

---

### 🧰 Real-world Example: Python (CPython)
- Due to the **Global Interpreter Lock (GIL)**, Python (CPython) threads don’t run in parallel on multiple cores — even though OS schedules them as normal threads.
- For true parallelism, Python uses **multiprocessing**, which uses **separate processes** — each with its own Python interpreter and memory space.

- **Each process has its own memory space**, resources, and **its own GIL** (in Python’s case), because they are isolated from each other.
- **Each process can be scheduled on a separate CPU core**, enabling *true parallelism* (unlike Python threads under the GIL).
- This is why `multiprocessing` in Python is preferred for **CPU-bound tasks**.

---

### ❓ **But here's where it gets interesting:**

#### ⚠️ **Can you run more processes than CPU cores?**
👉 **YES.** Absolutely. The OS **does not restrict** you to only run as many processes as there are physical cores.

#### 🔄 **How does it work then?**
When the number of processes > number of cores:
- The **OS schedules** them using **time slicing**.
- Each process gets a small time window (quantum) on the CPU.
- The OS **context switches** between processes, pausing and resuming them rapidly.
- From a user perspective, it feels like everything is happening "simultaneously" (thanks to the CPU's speed), but it's actually **interleaved execution**.

---

### 🧠 Example:
Suppose:
- You have **4 cores**, and
- You start **10 processes** via Python’s `multiprocessing`.

What happens:
- The OS will schedule 4 of them to run immediately (one per core).
- The remaining 6 will wait in a **ready queue**.
- As time progresses, the OS **rotates them** in and out of the CPU using context switches.

So:
- You get **concurrency**.
- But not all 10 are running **in parallel** — only 4 at a time can *actually* be executing.
- This can lead to some **overhead** due to context switching, especially if all processes are CPU-bound.

---

### ⚡ Performance Tip:
- For **CPU-bound tasks**: Use number of **processes = number of CPU cores** (can be checked via `os.cpu_count()`).
- For **I/O-bound tasks** (e.g., file/network read/write): You **can afford more** processes than cores, because I/O waits free up the CPU.

---

### 🧪 In Python:
```python
from multiprocessing import Pool, cpu_count

def task(x):
    # some CPU-bound operation
    return x * x

if __name__ == "__main__":
    with Pool(cpu_count()) as pool:
        results = pool.map(task, range(10))
    print(results)
```
Here `cpu_count()` ensures you're not oversubscribing CPU if it's CPU-bound. If I/O-bound, you might go beyond.

## 🔁 1. **Multiprocessing vs Threading for I/O-bound Tasks**

### 📌 Key Insight:
- **CPU-bound** tasks (like math computations) benefit from **multiprocessing** (bypass GIL).
- **I/O-bound** tasks (like reading files, API calls, DB access) benefit from **multithreading**, because while the thread is waiting (e.g., for a network response), another thread can run.

---

### ⚙️ Python Threading for I/O-bound Example
```python
import threading
import time

def io_bound_task(n):
    print(f"Start task {n}")
    time.sleep(2)  # Simulates I/O wait
    print(f"End task {n}")

threads = []

for i in range(5):
    t = threading.Thread(target=io_bound_task, args=(i,))
    t.start()
    threads.append(t)

for t in threads:
    t.join()
```

**Total time**: ~2 seconds (because threads run in parallel, even with GIL).

---

### ⚙️ Same Example with Multiprocessing
```python
from multiprocessing import Process
import time

def io_bound_task(n):
    print(f"Start task {n}")
    time.sleep(2)
    print(f"End task {n}")

processes = []

for i in range(5):
    p = Process(target=io_bound_task, args=(i,))
    p.start()
    processes.append(p)

for p in processes:
    p.join()
```

**Total time**: Also ~2 seconds. But **more resource-heavy** due to separate processes.

### ⚠️ Summary:
- For **I/O tasks**, **threading is cheaper** and just as effective.
- Avoid using multiprocessing for I/O-heavy workloads unless needed.

---

## 🔍 2. **CPU-bound Tasks: Threads vs Processes**

### CPU-bound Example with Threads
```python
import threading
import time

def cpu_task(n):
    count = 0
    for _ in range(10**7):
        count += 1
    print(f"Thread {n} done")

start = time.time()
threads = []

for i in range(4):  # Suppose 4 threads
    t = threading.Thread(target=cpu_task, args=(i,))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("Total time:", time.time() - start)
```

Due to the **GIL**, Python only runs **one thread at a time**, so performance isn't great.

---

### Same Task with Multiprocessing
```python
from multiprocessing import Process
import time

def cpu_task(n):
    count = 0
    for _ in range(10**7):
        count += 1
    print(f"Process {n} done")

start = time.time()
processes = []

for i in range(4):  # 4 CPU cores
    p = Process(target=cpu_task, args=(i,))
    p.start()
    processes.append(p)

for p in processes:
    p.join()

print("Total time:", time.time() - start)
```

Multiprocessing will utilize **all CPU cores**, so this will run **truly in parallel**, reducing total time.

---

## 📈 3. **How to Profile and Tune Multiprocess Programs**

### ✅ Use `time`, `psutil`, or `cProfile` for profiling:

```python
import time
from multiprocessing import Pool
import os, psutil

def cpu_task(x):
    total = 0
    for i in range(10**6):
        total += i * x
    return total

if __name__ == "__main__":
    start = time.time()
    with Pool(processes=os.cpu_count()) as pool:
        results = pool.map(cpu_task, range(10))
    print("Done in", time.time() - start)

    # Memory profiling
    print(f"Memory usage: {psutil.Process().memory_info().rss / (1024 * 1024)} MB")
```

---

### 💡 Tips to Tune:
| What to Tune                  | How                                  |
|-------------------------------|---------------------------------------|
| # of processes                | Use `os.cpu_count()`                 |
| Avoid shared state            | Use `Queue`, `Pipe`, `Value`, etc.  |
| Chunk large inputs            | Use `chunksize` in `pool.map()`     |
| Profile with `cProfile`       | For function-level breakdown         |
| Use `multiprocessing.dummy`  | A thread-based drop-in replacement   |

---

## 🔚 Summary Table

| Task Type  | Recommended Concurrency | Why                                     |
|------------|-------------------------|------------------------------------------|
| CPU-bound  | `multiprocessing`       | Bypasses GIL, uses multiple cores        |
| I/O-bound  | `threading` / `asyncio` | Threads free up CPU during I/O wait      |
| Mixed      | Hybrid (or async + workers) | Depends on dominant task type         |


Even and odd counter multithreading program
- Thread1 increments even
- Thread2 increments odd

In [5]:
import sys
sys._is_gil_enabled() 

True

In the given program, if you **don't use the `global` keyword** for the `counter` variable, you will get an error or unexpected behavior because of how **variable scope** works in Python.

### Why does the `global` keyword cause an error without it?

1. **Variable Scope**:
   - By default, when you modify a variable inside a function, Python assumes the variable is **local to that function**.
   - When you assign a value to a variable inside a function, Python treats it as a **local variable**. If the variable is not defined within the function, Python will try to look for it in the local scope. This can lead to errors when trying to modify a **global variable** from within a function without declaring it as `global`.

2. **What happens when `global` is not used**:
   - In your case, `counter` is a **global variable**, and both threads are trying to modify it. When you try to modify `counter` inside the thread functions (`thread1` and `thread2`), Python assumes you're trying to create a **local variable** called `counter` inside each function.
   - Python then can't find a reference to the global `counter` in the function's local scope, causing an **error** or **unexpected behavior** because it's trying to create a new local variable without any clear initialization.

### Example: What happens without `global`:

**Error:**
```
UnboundLocalError: local variable 'counter' referenced before assignment
```

### How does the `global` keyword fix the problem?

When you declare the variable as `global` inside the function, Python knows you want to refer to the global `counter` variable instead of creating a new local variable with the same name.

Now, when you use the `global` keyword, Python knows that you want to access and modify the global variable `counter`, and no error will occur.

### Summary:
- Without the `global` keyword, Python treats the `counter` variable as **local** to each thread function.
- Since `counter` is not defined locally within the function, Python throws an **UnboundLocalError**.
- Using the `global` keyword tells Python that you're referring to the **global variable**, not creating a new local one, and allows both threads to modify the same `counter` variable.

By marking `counter` as `global`, you ensure that the changes made in **both threads** are reflected on the same **global counter** and not on separate local copies of the variable.

No, simply declaring `global counter = 0` inside the function doesn't work. The `global` keyword is used to **refer to a global variable** inside a function, but **it cannot be used to define or initialize a variable**.

### Why does `global counter = 0` not work?

The `global` keyword is only used to **access** and **modify** an existing global variable inside a function. It does **not** allow you to declare or initialize the global variable.

Here's why:

1. **Purpose of `global` keyword**:
   - The `global` keyword is used to **refer to an already existing variable** in the global scope, not to initialize or assign a value to it.
   - When you use `global counter`, you are telling Python that you intend to modify or use the global `counter` variable, which already exists in the global scope.

2. **Declaring a new global variable**:
   - A variable must be initialized **once** in the global scope (outside of any functions) before you use it. You cannot **declare** a variable with `global` inside a function.

### What Happens if You Try `global counter = 0`?

If you try to do this:

```python
def thread1():
    global counter = 0  # This will give an error
    counter += 1
```

**Error**:
```
SyntaxError: can't use assignment in 'global' statement
```

### Explanation:
- The `global` keyword is used **without an assignment**, just to indicate that the variable is global. It doesn't allow you to assign a value to a global variable directly in the global scope inside a function.
- To initialize a global variable, you need to assign a value to it **before** using it in any functions. You can **initialize** it in the global scope, and then use `global` in functions when you want to **modify** it.

### Conclusion:

- You can **initialize** the global variable outside any function (e.g., `counter = 0`).
- You use the `global` keyword **inside the function** to modify the global variable.
- The `global` keyword cannot be used to **assign a value** to a global variable inside the function.


In [8]:
import threading
import time

counter = 0

#Thread1: Increment even counter
def even_counter():
    global counter # Use global to refer to the global 'counter'
    while counter < 10:
        if counter%2 == 0:
            print (f"Thread1(Even) - {counter}")
            counter+=1
        # time.sleep(0.1)  # Simulate some delay
        
def odd_counter():
    global counter
    while counter < 10:
        if counter%2 != 0:
            print (f"Thread1(Odd) - {counter}")
            counter+=1   
        # time.sleep(0.1)  # Simulate some delay

def main():
    #create 2 threads
    thread1 = threading.Thread(target=even_counter)
    thread2 = threading.Thread(target=odd_counter)

    # Start the threads
    thread1.start()
    thread2.start()

# Run the main function
if __name__ == "__main__":
    main()

Thread1(Even) - 0
Thread1(Odd) - 1
Thread1(Even) - 2
Thread1(Odd) - 3
Thread1(Even) - 4
Thread1(Odd) - 5
Thread1(Even) - 6
Thread1(Odd) - 7
Thread1(Even) - 8
Thread1(Odd) - 9
Thread1(Even) - 10


In [1]:
import threading


def print_cube(num):
    print("Cube: {}" .format(num * num * num))


def print_square(num):
    print("Square: {}" .format(num * num))


if __name__ =="__main__":
    t1 = threading.Thread(target=print_square, args=(10,))
    t2 = threading.Thread(target=print_cube, args=(10,))

    t1.start()
    t2.start()

    t1.join()
    t2.join()

    print("Done!")


Square: 100
Cube: 1000
Done!


In [2]:
import threading
import os

def task1():
    print("Task 1 assigned to thread: {}".format(threading.current_thread().name))
    print("ID of process running task 1: {}".format(os.getpid()))

def task2():
    print("Task 2 assigned to thread: {}".format(threading.current_thread().name))
    print("ID of process running task 2: {}".format(os.getpid()))

if __name__ == "__main__":

    print("ID of process running main program: {}".format(os.getpid()))

    print("Main thread name: {}".format(threading.current_thread().name))

    t1 = threading.Thread(target=task1, name='t1')
    t2 = threading.Thread(target=task2, name='t2')

    t1.start()
    t2.start()

    t1.join()
    t2.join()


ID of process running main program: 30744
Main thread name: MainThread
Task 1 assigned to thread: t1
ID of process running task 1: 30744
Task 2 assigned to thread: t2
ID of process running task 2: 30744


In [3]:
import concurrent.futures

def worker():
    print("Worker thread running")

pool = concurrent.futures.ThreadPoolExecutor(max_workers=2)

pool.submit(worker)
pool.submit(worker)

pool.shutdown(wait=True)

print("Main thread continuing to run")


Worker thread running
Worker thread running
Main thread continuing to run


In [12]:
#Race Condition example
import threading
import os

# global variable x 
x = 0

def increment(): 
	""" 
	function to increment global variable x 
	"""
	global x #explanation for this already given in above notes
	x += 1

def thread_task1(): 
	""" 
	task for thread 
	calls increment function 10 times. 
	"""
	print("Task assigned to thread1: {}".format(threading.current_thread().name))
	# print("ID of process running task 1 : {}".format(os.getpid()))
	for _ in range(10): 
		increment() 

def thread_task2(): 
	""" 
	task for thread 
	calls increment function 10 times. 
	"""
	print("Task assigned to thread2: {}".format(threading.current_thread().name))
	# print("ID of process running task 2 : {}".format(os.getpid()))
	for _ in range(10): 
		increment() 

def main_task(): 
	global x
	# print('I am X from main func',x)
	# setting global variable x as 0 
	# x = 0

	# creating threads 
	t1 = threading.Thread(target=thread_task1) 
	t2 = threading.Thread(target=thread_task2) 

	# start threads 
	t1.start() 
	t2.start() 

	# wait until threads finish their job 
	t1.join() 
	t2.join() 

if __name__ == "__main__": 
	for i in range(10): 
		main_task() 
		print("Iteration {0}: x = {1}".format(i,x)) 


Task assigned to thread1: Thread-145 (thread_task1)
Task assigned to thread2: Thread-146 (thread_task2)
Iteration 0: x = 20
Task assigned to thread1: Thread-147 (thread_task1)
Task assigned to thread2: Thread-148 (thread_task2)
Iteration 1: x = 40
Task assigned to thread1: Thread-149 (thread_task1)
Task assigned to thread2: Thread-150 (thread_task2)
Iteration 2: x = 60
Task assigned to thread1: Thread-151 (thread_task1)
Task assigned to thread2: Thread-152 (thread_task2)
Iteration 3: x = 80
Task assigned to thread1: Thread-153 (thread_task1)
Task assigned to thread2: Thread-154 (thread_task2)
Iteration 4: x = 100
Task assigned to thread1: Thread-155 (thread_task1)
Task assigned to thread2: Thread-156 (thread_task2)
Iteration 5: x = 120
Task assigned to thread1: Thread-157 (thread_task1)
Task assigned to thread2: Thread-158 (thread_task2)
Iteration 6: x = 140
Task assigned to thread1: Thread-159 (thread_task1)
Task assigned to thread2: Thread-160 (thread_task2)
Iteration 7: x = 160
Task